In [36]:
# Environment / path setup (kept minimal for agent execution)
from pathlib import Path
base_path = Path('.')
print('Base path set to current working directory.')

Base path set to current working directory.


# Insurance Analytics - Top 10 Business Questions

This notebook provides comprehensive analytics for key insurance business questions using actual policy and claims data.

## Table of Contents

| Question | Fabric Data Agent Question |
|----------|----------------------------|
| [Q1](#question-1-portfolio-mix-analysis) | What is the distribution of active policies by risk rating comparing 2023 vs 2024, and which risk segments show the largest percentage point changes? |
| [Q2](#question-2-pricing-alignment) | Are premium amounts monotonically increasing across risk ratings A through E, and what pricing inversions exist when adjusted for coverage amount? |
| [Q3](#question-3-coverage-adequacy) | Which 2024 policies have coverage-to-premium ratios more than 15% above their peer group median based on risk rating and age group? |
| [Q4](#question-4-product-mix-impact) | How do policy counts and average premiums by policy type compare between 2023 and 2024, and what are the market share changes? |
| [Q5](#question-5-aviation-analysis) | What are the differences in risk ratings, premiums, and claims performance between aviation-related occupations and non-aviation policies? |
| [Q6](#question-6-exam-exceptions) | How many high-risk (D/E rating) policies issued in 2024 did not require medical exams, and what are their characteristics? |
| [Q7](#question-7-occupation-anomalies) | Which occupations with at least 5 claims in 2024 have denial rates significantly different from the overall denial rate? |
| [Q8](#question-8-claims-trends) | What are the annual and quarterly trends in claim frequency and average claim amounts, particularly for 2024? |
| [Q9](#question-9-preexisting-conditions) | How do pre-existing conditions impact claim denial rates and average claim amounts compared to claims without pre-existing conditions? |
| [Q10](#question-10-data-quality) | What percentage of critical fields are missing in both policy and claims data, and how does this affect data completeness scores? |

---

**Data Sources:**
- `policy_portfolio.csv` - 890 policies with 24 fields (premiums, coverage, risk ratings, demographics)
- `claims_portfolio.csv` - 236 claims with 14 fields (amounts, status, causes, dates)

**Analysis Period:** 2020-2025 data with focus on 2024 YTD and 2023 comparisons

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
#import duckdb
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load Insurance Data
try:
    # Load policy and claims data
    policies_df = pd.read_csv('case_study/data/policy_portfolio.csv')
    claims_df = pd.read_csv('case_study/data/claims_portfolio.csv')
    
    print("Data loaded successfully!")
    print(f"Policies dataset shape: {policies_df.shape}")
    print(f"Claims dataset shape: {claims_df.shape}")
    
    # Convert date columns
    policies_df['issue_date'] = pd.to_datetime(policies_df['issue_date'])
    policies_df['last_premium_paid_date'] = pd.to_datetime(policies_df['last_premium_paid_date'])
    claims_df['claim_date'] = pd.to_datetime(claims_df['claim_date'])
    
    # Extract year and quarter information
    policies_df['issue_year'] = policies_df['issue_date'].dt.year
    policies_df['issue_quarter'] = policies_df['issue_date'].dt.quarter
    claims_df['claim_year'] = claims_df['claim_date'].dt.year
    claims_df['claim_quarter'] = claims_df['claim_date'].dt.quarter
    
    print("Date conversions completed!")
    
except FileNotFoundError as e:
    print(f"Error loading data: {e}")
    print("Please ensure the CSV files are in the correct location.")

Data loaded successfully!
Policies dataset shape: (890, 24)
Claims dataset shape: (236, 14)
Date conversions completed!


---

# Analysis Implementation

Each question below provides a complete analysis with business insights and recommendations.

## Question 1: Portfolio Mix Analysis {#question-1-portfolio-mix-analysis}

**Question:** What is the distribution of active policies by risk rating comparing 2023 vs 2024, and which risk segments show the largest percentage point changes?

**Python Code to Answer:**

In [39]:
# Question 1: Portfolio Mix Analysis
print("=" * 60)
print("QUESTION 1: PORTFOLIO MIX ANALYSIS")
print("=" * 60)

# Filter active policies for comparison years
active_2023 = policies_df[(policies_df['issue_year'] == 2023) & (policies_df['policy_status'] == 'Active')]
active_2024 = policies_df[(policies_df['issue_year'] == 2024) & (policies_df['policy_status'] == 'Active')]

print(f"\nData Overview:")
print(f"• Active policies issued in 2023: {len(active_2023):,}")
print(f"• Active policies issued in 2024: {len(active_2024):,}")

# Calculate risk distribution for both years
def calculate_risk_distribution(df, year):
    """Calculate risk rating distribution with percentages"""
    risk_dist = df.groupby('risk_rating').agg({
        'policy_id': 'count',
        'coverage_amount': 'sum'
    }).round(0)
    risk_dist.columns = ['Policy_Count', 'Total_Coverage']
    risk_dist['Policy_Pct'] = (risk_dist['Policy_Count'] / risk_dist['Policy_Count'].sum() * 100).round(1)
    risk_dist['Coverage_Pct'] = (risk_dist['Total_Coverage'] / risk_dist['Total_Coverage'].sum() * 100).round(1)
    return risk_dist

# Generate distributions
dist_2023 = calculate_risk_distribution(active_2023, 2023)
dist_2024 = calculate_risk_distribution(active_2024, 2024)

print(f"\n📊 2023 Risk Distribution:")
print(dist_2023)

print(f"\n📊 2024 Risk Distribution:")
print(dist_2024)

# Calculate percentage point changes
print(f"\n📈 Portfolio Mix Changes (2023 → 2024):")
all_risks = set(dist_2023.index) | set(dist_2024.index)
changes = []
for risk in sorted(all_risks):
    pct_2023 = dist_2023.loc[risk, 'Policy_Pct'] if risk in dist_2023.index else 0
    pct_2024 = dist_2024.loc[risk, 'Policy_Pct'] if risk in dist_2024.index else 0
    change = pct_2024 - pct_2023
    changes.append({'Risk_Rating': risk, '2023_Pct': pct_2023, '2024_Pct': pct_2024, 'Change_pp': change})

changes_df = pd.DataFrame(changes).sort_values('Change_pp', ascending=False)
print(changes_df)

# Key insights
largest_increase = changes_df.iloc[0]
largest_decrease = changes_df.iloc[-1]

print(f"\n🎯 Key Insights:")
print(f"• Largest increase: Risk {largest_increase['Risk_Rating']} (+{largest_increase['Change_pp']:.1f} percentage points)")
print(f"• Largest decrease: Risk {largest_decrease['Risk_Rating']} ({largest_decrease['Change_pp']:.1f} percentage points)")

# Coverage concentration
if len(active_2024) > 0:
    high_risk_coverage_2024 = active_2024[active_2024['risk_rating'].isin(['D', 'E'])]['coverage_amount'].sum()
    total_coverage_2024 = active_2024['coverage_amount'].sum()
    high_risk_pct = (high_risk_coverage_2024 / total_coverage_2024 * 100).round(1)
    print(f"• High-risk (D+E) coverage concentration in 2024: {high_risk_pct}%")

print(f"\n💡 Business Recommendation:")
print(f"Monitor portfolio drift toward higher-risk segments and ensure adequate pricing/reserves.")

QUESTION 1: PORTFOLIO MIX ANALYSIS

Data Overview:
• Active policies issued in 2023: 81
• Active policies issued in 2024: 12

📊 2023 Risk Distribution:
             Policy_Count  Total_Coverage  Policy_Pct  Coverage_Pct
risk_rating                                                        
A                      37      24340000.0        45.7          50.3
B                      15       6735000.0        18.5          13.9
C                       8       7040000.0         9.9          14.6
D                      18       9140000.0        22.2          18.9
E                       3       1110000.0         3.7           2.3

📊 2024 Risk Distribution:
             Policy_Count  Total_Coverage  Policy_Pct  Coverage_Pct
risk_rating                                                        
A                       2        580000.0        16.7           6.2
C                       4       3820000.0        33.3          40.9
D                       5       4050000.0        41.7          43.3
E    

**SQL Code to Answer:**

In [6]:
# Question 1: Portfolio Mix Analysis - SQL Implementation (Single Query)
import duckdb

# Create connection and load data into SQL tables
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE policies AS SELECT * FROM policies_df")

print("=" * 60)
print("QUESTION 1: PORTFOLIO MIX ANALYSIS (SQL - SINGLE QUERY)")
print("=" * 60)

# Comprehensive SQL Query: All portfolio analysis in one query
comprehensive_query = """
WITH yearly_stats AS (
    -- Base statistics by year and risk rating
    SELECT 
        EXTRACT(YEAR FROM issue_date) as year,
        risk_rating,
        COUNT(*) as policy_count,
        SUM(coverage_amount) as total_coverage,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY EXTRACT(YEAR FROM issue_date)), 1) as policy_pct,
        ROUND(SUM(coverage_amount) * 100.0 / SUM(SUM(coverage_amount)) OVER (PARTITION BY EXTRACT(YEAR FROM issue_date)), 1) as coverage_pct
    FROM policies 
    WHERE EXTRACT(YEAR FROM issue_date) IN (2023, 2024) 
        AND policy_status = 'Active'
    GROUP BY EXTRACT(YEAR FROM issue_date), risk_rating
),
risk_comparison AS (
    -- Pivot data for year-over-year comparison
    SELECT 
        risk_rating,
        SUM(CASE WHEN year = 2023 THEN policy_count ELSE 0 END) as count_2023,
        SUM(CASE WHEN year = 2024 THEN policy_count ELSE 0 END) as count_2024,
        SUM(CASE WHEN year = 2023 THEN total_coverage ELSE 0 END) as coverage_2023,
        SUM(CASE WHEN year = 2024 THEN total_coverage ELSE 0 END) as coverage_2024,
        AVG(CASE WHEN year = 2023 THEN policy_pct END) as pct_2023,
        AVG(CASE WHEN year = 2024 THEN policy_pct END) as pct_2024
    FROM yearly_stats
    GROUP BY risk_rating
),
high_risk_stats AS (
    -- High-risk concentration for 2024
    SELECT 
        ROUND(
            SUM(CASE WHEN risk_rating IN ('D', 'E') THEN coverage_amount ELSE 0 END) * 100.0 / 
            SUM(coverage_amount), 1
        ) as high_risk_pct_2024
    FROM policies 
    WHERE EXTRACT(YEAR FROM issue_date) = 2024 AND policy_status = 'Active'
)
-- Final comprehensive result
SELECT 
    rc.risk_rating,
    rc.count_2023,
    rc.count_2024,
    (rc.count_2024 - rc.count_2023) as count_change,
    COALESCE(rc.pct_2023, 0) as pct_2023,
    COALESCE(rc.pct_2024, 0) as pct_2024,
    ROUND(COALESCE(rc.pct_2024, 0) - COALESCE(rc.pct_2023, 0), 1) as change_pp,
    ROUND(rc.coverage_2023, 0) as coverage_2023,
    ROUND(rc.coverage_2024, 0) as coverage_2024,
    hrs.high_risk_pct_2024
FROM risk_comparison rc
CROSS JOIN high_risk_stats hrs
WHERE rc.count_2023 > 0 OR rc.count_2024 > 0
ORDER BY change_pp DESC;
"""

print("\n📊 Comprehensive Portfolio Analysis (SQL):")
result = conn.execute(comprehensive_query).fetchdf()
print(result)

# Extract key insights from the single query result
if not result.empty:
    print(f"\n🎯 Key Insights from Single Query:")
    
    # Largest changes
    largest_increase = result.iloc[0]
    largest_decrease = result.iloc[-1]
    
    print(f"• Largest increase: Risk {largest_increase['risk_rating']} (+{largest_increase['change_pp']:.1f} percentage points)")
    print(f"• Largest decrease: Risk {largest_decrease['risk_rating']} ({largest_decrease['change_pp']:.1f} percentage points)")
    
    # High-risk concentration (same for all rows, so take first)
    high_risk_pct = result['high_risk_pct_2024'].iloc[0]
    print(f"• High-risk (D+E) coverage concentration in 2024: {high_risk_pct}%")

# Close connection
conn.close()

print(f"\n💡 Business Recommendation:")
print(f"SQL analysis confirms portfolio drift patterns. Monitor risk segment changes and adjust pricing/reserves accordingly.")

QUESTION 1: PORTFOLIO MIX ANALYSIS (SQL - SINGLE QUERY)

📊 Comprehensive Portfolio Analysis (SQL):
  risk_rating  count_2023  count_2024  count_change  pct_2023  pct_2024  \
0           C         8.0         4.0          -4.0       9.9      33.3   
1           D        18.0         5.0         -13.0      22.2      41.7   
2           E         3.0         1.0          -2.0       3.7       8.3   
3           B        15.0         0.0         -15.0      18.5       0.0   
4           A        37.0         2.0         -35.0      45.7      16.7   

   change_pp  coverage_2023  coverage_2024  high_risk_pct_2024  
0       23.4      7040000.0      3820000.0                52.9  
1       19.5      9140000.0      4050000.0                52.9  
2        4.6      1110000.0       900000.0                52.9  
3      -18.5      6735000.0            0.0                52.9  
4      -29.0     24340000.0       580000.0                52.9  

🎯 Key Insights from Single Query:
• Largest increase: Risk 

## Question 2: Pricing Alignment {#question-2-pricing-alignment}

**Question:** Are premium amounts monotonically increasing across risk ratings A through E, and what pricing inversions exist when adjusted for coverage amount?

**Python Code to Answer:**

In [40]:
# Question 2: Pricing Alignment
print("=" * 60)
print("QUESTION 2: PRICING ALIGNMENT")
print("=" * 60)

# Calculate pricing metrics by risk rating
pricing_stats = policies_df.groupby('risk_rating').agg({
    'premium': ['mean', 'median', 'std', 'count'],
    'coverage_amount': 'mean'
}).round(2)

# Flatten column names
pricing_stats.columns = ['Avg_Premium', 'Median_Premium', 'Std_Premium', 'Policy_Count', 'Avg_Coverage']

# Calculate risk-adjusted pricing (premium per $1000 coverage)
pricing_stats['Premium_per_1K'] = (pricing_stats['Avg_Premium'] / pricing_stats['Avg_Coverage'] * 1000).round(3)

print(f"\n📊 Pricing Analysis by Risk Rating:")
print(pricing_stats)

# Test monotonicity - premiums should increase with risk
risk_order = ['A', 'B', 'C', 'D', 'E']
available_risks = [r for r in risk_order if r in pricing_stats.index]

if len(available_risks) > 1:
    print(f"\n🔍 Monotonicity Test:")
    print(f"Risk progression: {' → '.join(available_risks)}")
    
    # Check raw premiums
    premiums = [pricing_stats.loc[risk, 'Avg_Premium'] for risk in available_risks]
    premium_monotonic = all(premiums[i] <= premiums[i+1] for i in range(len(premiums)-1))
    
    # Check risk-adjusted premiums
    risk_adj_premiums = [pricing_stats.loc[risk, 'Premium_per_1K'] for risk in available_risks]
    risk_adj_monotonic = all(risk_adj_premiums[i] <= risk_adj_premiums[i+1] for i in range(len(risk_adj_premiums)-1))
    
    print(f"\nRaw premiums: {' → '.join([f'${p:,.0f}' for p in premiums])}")
    print(f"Monotonic: {'✅ YES' if premium_monotonic else '❌ NO'}")
    
    print(f"\nRisk-adjusted (per $1K): {' → '.join([f'${p:.2f}' for p in risk_adj_premiums])}")
    print(f"Monotonic: {'✅ YES' if risk_adj_monotonic else '❌ NO'}")
    
    # Identify inversions
    inversions = []
    for i in range(len(available_risks)-1):
        current_risk = available_risks[i]
        next_risk = available_risks[i+1]
        if risk_adj_premiums[i] > risk_adj_premiums[i+1]:
            inversions.append(f"{current_risk} > {next_risk} (${risk_adj_premiums[i]:.2f} > ${risk_adj_premiums[i+1]:.2f} per $1K)")
    
    print(f"\n⚠️  Pricing Inversions Found:")
    if inversions:
        for inversion in inversions:
            print(f"• {inversion}")
    else:
        print("• None - pricing structure is properly aligned")

print(f"\n💡 Business Recommendation:")
if inversions:
    print(f"Review pricing for risk categories with inversions. Consider repricing {len(inversions)} identified issues.")
else:
    print(f"Pricing structure appears well-aligned with risk levels. Continue monitoring.")

QUESTION 2: PRICING ALIGNMENT

📊 Pricing Analysis by Risk Rating:
             Avg_Premium  Median_Premium  Std_Premium  Policy_Count  \
risk_rating                                                           
A                1908.95           662.0      3293.03           173   
B                4050.45          1455.1      7884.43           217   
C                2217.70          1245.0      2755.96           184   
D                1649.89          1111.5      2091.41           184   
E                1649.76          1382.5      1418.58           132   

             Avg_Coverage  Premium_per_1K  
risk_rating                                
A               867774.57           2.200  
B               997811.06           4.059  
C               628831.52           3.527  
D               384347.83           4.293  
E               285318.18           5.782  

🔍 Monotonicity Test:
Risk progression: A → B → C → D → E

Raw premiums: $1,909 → $4,050 → $2,218 → $1,650 → $1,650
Monotonic: ❌

**SQL Code to Answer:**

In [13]:
# Question 2: Pricing Alignment - SQL Implementation (Single Query)
import duckdb

# Create fresh connection and load data
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE policies AS SELECT * FROM policies_df")

print("=" * 60)
print("QUESTION 2: PRICING ALIGNMENT (SQL - SINGLE QUERY)")
print("=" * 60)

# Compact SQL Query: All pricing analysis in one streamlined query
pricing_query = """
WITH pricing_stats AS (
    SELECT 
        risk_rating,
        COUNT(*) as policy_count,
        ROUND(AVG(premium), 2) as avg_premium,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY premium), 2) as median_premium,
        ROUND(AVG(coverage_amount), 2) as avg_coverage,
        ROUND(AVG(premium) / AVG(coverage_amount) * 1000, 3) as premium_per_1k
    FROM policies GROUP BY risk_rating
),
monotonicity AS (
    SELECT 
        p1.risk_rating || ' → ' || p2.risk_rating as progression,
        '$' || ROUND(p1.premium_per_1k, 2) || ' → $' || ROUND(p2.premium_per_1k, 2) || ' per $1K' as rate_progression,
        CASE WHEN p1.premium_per_1k <= p2.premium_per_1k THEN 'YES' ELSE 'NO' END as monotonic
    FROM pricing_stats p1 
    JOIN pricing_stats p2 ON 
        (p1.risk_rating = 'A' AND p2.risk_rating = 'B') OR
        (p1.risk_rating = 'B' AND p2.risk_rating = 'C') OR  
        (p1.risk_rating = 'C' AND p2.risk_rating = 'D') OR
        (p1.risk_rating = 'D' AND p2.risk_rating = 'E')
),
combined_results AS (
    SELECT 
        1 as sort_order,
        'STATS' as section, 
        risk_rating as detail, 
        policy_count, 
        avg_premium, 
        median_premium, 
        avg_coverage, 
        premium_per_1k,
        NULL as status
    FROM pricing_stats
    UNION ALL
    SELECT 
        2 as sort_order,
        'MONOTONICITY' as section,
        progression as detail,
        NULL, NULL, NULL, NULL, NULL,
        rate_progression || ' - ' || monotonic as status
    FROM monotonicity
    UNION ALL
    SELECT 
        3 as sort_order,
        'SUMMARY' as section,
        'Assessment' as detail,
        NULL, NULL, NULL, NULL, NULL,
        CASE WHEN (SELECT COUNT(*) FROM monotonicity WHERE monotonic = 'NO') > 0 
            THEN 'Found ' || (SELECT COUNT(*) FROM monotonicity WHERE monotonic = 'NO') || ' pricing inversions'
            ELSE 'No inversions - pricing aligned'
        END as status
)
SELECT section, detail, policy_count, avg_premium, median_premium, avg_coverage, premium_per_1k, status
FROM combined_results
ORDER BY sort_order, detail;
"""

print("\n📊 Comprehensive Pricing Analysis (SQL):")
result = conn.execute(pricing_query).fetchdf()

# Display results by section
stats = result[result['section'] == 'STATS']
mono = result[result['section'] == 'MONOTONICITY']
summary = result[result['section'] == 'SUMMARY']

print("\n📈 Pricing Statistics:")
print(stats[['detail', 'policy_count', 'avg_premium', 'median_premium', 'premium_per_1k']].to_string(index=False))

print("\n🔍 Monotonicity Check:")
for _, row in mono.iterrows():
    print(f"• {row['detail']}: {row['status']}")

print(f"\n⚠️  Assessment:")
print(f"• {summary['status'].iloc[0]}")

# Identify inversions
inversions = mono[mono['status'].str.contains('NO')]
if len(inversions) > 0:
    print(f"\n🚨 Pricing Inversions:")
    for _, row in inversions.iterrows():
        print(f"  - {row['detail']}")

conn.close()

print(f"\n💡 Business Recommendation:")
if len(inversions) > 0:
    print(f"Review {len(inversions)} pricing inversion(s) for risk-based repricing.")
else:
    print(f"Pricing structure is well-aligned with risk levels.")

QUESTION 2: PRICING ALIGNMENT (SQL - SINGLE QUERY)

📊 Comprehensive Pricing Analysis (SQL):

📈 Pricing Statistics:
detail  policy_count  avg_premium  median_premium  premium_per_1k
     A           173      1908.95           662.0           2.200
     B           217      4050.45          1455.1           4.059
     C           184      2217.70          1245.0           3.527
     D           184      1649.89          1111.5           4.293
     E           132      1649.76          1382.5           5.782

🔍 Monotonicity Check:
• A → B: $2.2 → $4.06 per $1K - YES
• B → C: $4.06 → $3.53 per $1K - NO
• C → D: $3.53 → $4.29 per $1K - YES
• D → E: $4.29 → $5.78 per $1K - YES

⚠️  Assessment:
• Found 1 pricing inversions

🚨 Pricing Inversions:
  - B → C

💡 Business Recommendation:
Review 1 pricing inversion(s) for risk-based repricing.


## Question 3: Coverage Adequacy {#question-3-coverage-adequacy}

**Question:** Which 2024 policies have coverage-to-premium ratios more than 15% above their peer group median based on risk rating and age group?

**Python Code to Answer:**

In [41]:
# Question 3: Coverage Adequacy
print("=" * 60)
print("QUESTION 3: COVERAGE ADEQUACY")
print("=" * 60)

# Focus on 2024 policies for adequacy analysis
policies_2024 = policies_df[policies_df['issue_year'] == 2024].copy()

if len(policies_2024) > 0:
    # Calculate coverage-to-premium ratio
    policies_2024['coverage_premium_ratio'] = policies_2024['coverage_amount'] / policies_2024['premium']
    
    # Create age groups for peer comparison
    policies_2024['age_group'] = pd.cut(policies_2024['age'], 
                                       bins=[0, 30, 40, 50, 60, 100], 
                                       labels=['<30', '30-39', '40-49', '50-59', '60+'])
    
    print(f"📊 Analyzing {len(policies_2024)} policies issued in 2024")
    
    # Calculate peer group medians
    peer_medians = policies_2024.groupby(['risk_rating', 'age_group'])['coverage_premium_ratio'].median()
    
    # Find outliers (>15% above peer median)
    outliers = []
    for idx, policy in policies_2024.iterrows():
        try:
            peer_median = peer_medians.loc[(policy['risk_rating'], policy['age_group'])]
            deviation_pct = ((policy['coverage_premium_ratio'] - peer_median) / peer_median * 100)
            
            if deviation_pct > 15:  # Significant outlier
                outliers.append({
                    'policy_id': policy['policy_id'],
                    'risk_rating': policy['risk_rating'],
                    'age_group': policy['age_group'],
                    'coverage_amount': policy['coverage_amount'],
                    'premium': policy['premium'],
                    'ratio': policy['coverage_premium_ratio'],
                    'peer_median': peer_median,
                    'deviation_pct': deviation_pct
                })
        except KeyError:
            continue  # Skip if no peer group
    
    print(f"\n🔍 Coverage Adequacy Analysis:")
    print(f"• Total policies analyzed: {len(policies_2024):,}")
    print(f"• Potential over-insurance cases (>15% above peers): {len(outliers)}")
    
    if outliers:
        outliers_df = pd.DataFrame(outliers)
        outliers_df = outliers_df.sort_values('deviation_pct', ascending=False)
        
        print(f"\n⚠️  Top Over-Insurance Cases:")
        top_cases = outliers_df.head(5)[['policy_id', 'risk_rating', 'coverage_amount', 'premium', 'deviation_pct']]
        print(top_cases.round(1))
        
        # Summary statistics
        total_excess_coverage = (outliers_df['coverage_amount'] - 
                               (outliers_df['peer_median'] * outliers_df['premium'])).sum()
        
        print(f"\n📈 Summary Statistics:")
        print(f"• Average deviation: {outliers_df['deviation_pct'].mean():.1f}%")
        print(f"• Maximum deviation: {outliers_df['deviation_pct'].max():.1f}%")
        print(f"• Total excess coverage amount: ${total_excess_coverage:,.0f}")
    
    else:
        print("• No significant over-insurance cases identified")
        
else:
    print("❌ No 2024 policies available for analysis")

print(f"\n💡 Business Recommendation:")
print(f"Review flagged cases for appropriate coverage levels and potential fraud indicators.")

QUESTION 3: COVERAGE ADEQUACY
📊 Analyzing 13 policies issued in 2024

🔍 Coverage Adequacy Analysis:
• Total policies analyzed: 13
• Potential over-insurance cases (>15% above peers): 1

⚠️  Top Over-Insurance Cases:
  policy_id risk_rating  coverage_amount  premium  deviation_pct
0  LIF-1225           A         400000.0    312.0           15.5

📈 Summary Statistics:
• Average deviation: 15.5%
• Maximum deviation: 15.5%
• Total excess coverage amount: $53,750

💡 Business Recommendation:
Review flagged cases for appropriate coverage levels and potential fraud indicators.


**SQL Code to Answer:**

In [21]:
# Question 3: Coverage Adequacy - SQL Implementation (Single Query)
import duckdb

# Create fresh connection and load data
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE policies AS SELECT * FROM policies_df")

print("=" * 60)
print("QUESTION 3: COVERAGE ADEQUACY (SQL - SINGLE QUERY)")
print("=" * 60)

# Compact SQL Query: Coverage adequacy analysis in one query
coverage_query = """
WITH peer_analysis AS (
    SELECT 
        policy_id, risk_rating, coverage_amount, premium,
        coverage_amount / premium as ratio,
        MEDIAN(coverage_amount / premium) 
        OVER (PARTITION BY risk_rating, 
              CASE WHEN age < 30 THEN '<30' WHEN age < 40 THEN '30-39' 
                   WHEN age < 50 THEN '40-49' WHEN age < 60 THEN '50-59' ELSE '60+' END) as peer_median
    FROM policies WHERE issue_year = 2024
)
SELECT 
    policy_id, risk_rating, coverage_amount, premium,
    ROUND(ratio, 2) as ratio,
    ROUND(peer_median, 2) as peer_median,
    ROUND((ratio - peer_median) / peer_median * 100, 1) as deviation_pct
FROM peer_analysis 
WHERE (ratio - peer_median) / peer_median > 0.15
ORDER BY deviation_pct DESC;
"""

print("\n📊 Coverage Adequacy Analysis (SQL):")
result = conn.execute(coverage_query).fetchdf()

# Get total 2024 policies for context
total_2024 = conn.execute("SELECT COUNT(*) as count FROM policies WHERE issue_year = 2024").fetchdf()['count'].iloc[0]

print(f"\n📈 Analysis Summary:")
print(f"• Total 2024 Policies: {total_2024:,}")
print(f"• Over-Insurance Cases (>15%): {len(result):,}")

if len(result) > 0:
    print(f"\n⚠️  Top Over-Insurance Cases:")
    top_cases = result.head(5)[['policy_id', 'risk_rating', 'coverage_amount', 'premium', 'deviation_pct']]
    print(top_cases.to_string(index=False))
    
    print(f"\n📊 Over-Insurance Statistics:")
    print(f"• Average deviation: {result['deviation_pct'].mean():.1f}%")
    print(f"• Maximum deviation: {result['deviation_pct'].max():.1f}%")
    
    # Calculate excess coverage
    excess_coverage = (result['coverage_amount'] - (result['peer_median'] * result['premium'])).sum()
    print(f"• Total excess coverage: ${excess_coverage:,.0f}")
else:
    print("\n✅ No significant over-insurance cases identified")

conn.close()

print(f"\n💡 Business Recommendation:")
print(f"Review flagged cases for appropriate coverage levels and potential fraud indicators.")

QUESTION 3: COVERAGE ADEQUACY (SQL - SINGLE QUERY)

📊 Coverage Adequacy Analysis (SQL):

📈 Analysis Summary:
• Total 2024 Policies: 13
• Over-Insurance Cases (>15%): 1

⚠️  Top Over-Insurance Cases:
policy_id risk_rating  coverage_amount  premium  deviation_pct
 LIF-1225           A         400000.0    312.0           15.5

📊 Over-Insurance Statistics:
• Average deviation: 15.5%
• Maximum deviation: 15.5%
• Total excess coverage: $53,749

💡 Business Recommendation:
Review flagged cases for appropriate coverage levels and potential fraud indicators.


## Question 4: Product Mix Impact {#question-4-product-mix-impact}

**Question:** How do policy counts and average premiums by policy type compare between 2023 and 2024, and what are the market share changes?

**Python Code to Answer:**

In [42]:
# Question 4: Product Mix Impact
print("=" * 60)
print("QUESTION 4: PRODUCT MIX IMPACT")
print("=" * 60)

# Analyze product performance by year
def analyze_product_mix(df, year):
    """Analyze product mix for a given year"""
    return df.groupby('policy_type').agg({
        'policy_id': 'count',
        'premium': 'mean',
        'coverage_amount': 'mean'
    }).round(0)

# Get product mix for both years
products_2023 = analyze_product_mix(policies_df[policies_df['issue_year'] == 2023], 2023)
products_2024 = analyze_product_mix(policies_df[policies_df['issue_year'] == 2024], 2024)

products_2023.columns = ['Count_2023', 'Avg_Premium_2023', 'Avg_Coverage_2023']
products_2024.columns = ['Count_2024', 'Avg_Premium_2024', 'Avg_Coverage_2024']

# Combine data
product_comparison = pd.concat([products_2023, products_2024], axis=1).fillna(0)

# Calculate changes
product_comparison['Count_Change'] = product_comparison['Count_2024'] - product_comparison['Count_2023']
product_comparison['Count_Change_Pct'] = ((product_comparison['Count_2024'] - product_comparison['Count_2023']) / 
                                         product_comparison['Count_2023'] * 100).round(1)

# Handle division by zero
product_comparison['Premium_Change_Pct'] = np.where(
    product_comparison['Avg_Premium_2023'] > 0,
    ((product_comparison['Avg_Premium_2024'] - product_comparison['Avg_Premium_2023']) / 
     product_comparison['Avg_Premium_2023'] * 100).round(1),
    np.nan
)

print(f"📊 Product Mix Comparison (2023 vs 2024):")
print(product_comparison[['Count_2023', 'Count_2024', 'Count_Change', 'Count_Change_Pct']])

print(f"\n💰 Premium Analysis:")
print(product_comparison[['Avg_Premium_2023', 'Avg_Premium_2024', 'Premium_Change_Pct']])

# Market share analysis
total_2023 = product_comparison['Count_2023'].sum()
total_2024 = product_comparison['Count_2024'].sum()

if total_2023 > 0 and total_2024 > 0:
    print(f"\n📈 Market Share Analysis:")
    print(f"Total policies - 2023: {total_2023:.0f}, 2024: {total_2024:.0f}")
    
    for product in product_comparison.index:
        share_2023 = (product_comparison.loc[product, 'Count_2023'] / total_2023 * 100).round(1)
        share_2024 = (product_comparison.loc[product, 'Count_2024'] / total_2024 * 100).round(1)
        share_change = share_2024 - share_2023
        print(f"• {product}: {share_2023}% → {share_2024}% ({share_change:+.1f}pp)")

# Identify trends
declining_products = product_comparison[product_comparison['Count_Change'] < 0].index.tolist()
growing_products = product_comparison[product_comparison['Count_Change'] > 0].index.tolist()

print(f"\n🎯 Key Insights:")
if growing_products:
    print(f"• Growing products: {', '.join(growing_products)}")
if declining_products:
    print(f"• Declining products: {', '.join(declining_products)}")

print(f"\n💡 Business Recommendation:")
print(f"Focus marketing efforts on growing segments and review strategy for declining products.")

QUESTION 4: PRODUCT MIX IMPACT
📊 Product Mix Comparison (2023 vs 2024):
                Count_2023  Count_2024  Count_Change  Count_Change_Pct
policy_type                                                           
Term Life               79        12.0         -67.0             -84.8
Universal Life           5         1.0          -4.0             -80.0
Whole Life               8         0.0          -8.0            -100.0

💰 Premium Analysis:
                Avg_Premium_2023  Avg_Premium_2024  Premium_Change_Pct
policy_type                                                           
Term Life                 1146.0            1613.0                40.8
Universal Life            4046.0             312.0               -92.3
Whole Life                1366.0               0.0              -100.0

📈 Market Share Analysis:
Total policies - 2023: 92, 2024: 13
• Term Life: 85.9% → 92.3% (+6.4pp)
• Universal Life: 5.4% → 7.7% (+2.3pp)
• Whole Life: 8.7% → 0.0% (-8.7pp)

🎯 Key Insights:
• Declin

**SQL Code to Answer:**

In [22]:
# Question 4: Product Mix Impact - SQL Implementation (Single Query)
import duckdb

# Create fresh connection and load data
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE policies AS SELECT * FROM policies_df")

print("=" * 60)
print("QUESTION 4: PRODUCT MIX IMPACT (SQL - SINGLE QUERY)")
print("=" * 60)

# Compact SQL Query: Product mix analysis in one query
product_mix_query = """
WITH yearly_stats AS (
    SELECT 
        policy_type,
        SUM(CASE WHEN issue_year = 2023 THEN 1 ELSE 0 END) as count_2023,
        SUM(CASE WHEN issue_year = 2024 THEN 1 ELSE 0 END) as count_2024,
        ROUND(AVG(CASE WHEN issue_year = 2023 THEN premium END), 0) as avg_premium_2023,
        ROUND(AVG(CASE WHEN issue_year = 2024 THEN premium END), 0) as avg_premium_2024
    FROM policies 
    WHERE issue_year IN (2023, 2024)
    GROUP BY policy_type
),
market_share AS (
    SELECT 
        policy_type,
        count_2023, count_2024,
        (count_2024 - count_2023) as count_change,
        ROUND((count_2024 - count_2023) * 100.0 / NULLIF(count_2023, 0), 1) as count_change_pct,
        avg_premium_2023, avg_premium_2024,
        ROUND((avg_premium_2024 - avg_premium_2023) * 100.0 / NULLIF(avg_premium_2023, 0), 1) as premium_change_pct,
        ROUND(count_2023 * 100.0 / SUM(count_2023) OVER(), 1) as share_2023,
        ROUND(count_2024 * 100.0 / SUM(count_2024) OVER(), 1) as share_2024
    FROM yearly_stats
    WHERE count_2023 > 0 OR count_2024 > 0
)
SELECT 
    policy_type,
    count_2023, count_2024, count_change, count_change_pct,
    avg_premium_2023, avg_premium_2024, premium_change_pct,
    share_2023, share_2024,
    ROUND(share_2024 - share_2023, 1) as share_change_pp
FROM market_share
ORDER BY count_change DESC;
"""

print("\n📊 Product Mix Analysis (SQL):")
result = conn.execute(product_mix_query).fetchdf()

print("\n📈 Product Mix Comparison (2023 vs 2024):")
display_cols = ['policy_type', 'count_2023', 'count_2024', 'count_change', 'count_change_pct']
print(result[display_cols].to_string(index=False))

print(f"\n💰 Premium Analysis:")
premium_cols = ['policy_type', 'avg_premium_2023', 'avg_premium_2024', 'premium_change_pct']
print(result[premium_cols].to_string(index=False))

print(f"\n📈 Market Share Analysis:")
share_cols = ['policy_type', 'share_2023', 'share_2024', 'share_change_pp']
for _, row in result[share_cols].iterrows():
    print(f"• {row['policy_type']}: {row['share_2023']}% → {row['share_2024']}% ({row['share_change_pp']:+.1f}pp)")

# Identify trends
growing_products = result[result['count_change'] > 0]['policy_type'].tolist()
declining_products = result[result['count_change'] < 0]['policy_type'].tolist()

print(f"\n🎯 Key Insights:")
if growing_products:
    print(f"• Growing products: {', '.join(growing_products)}")
if declining_products:
    print(f"• Declining products: {', '.join(declining_products)}")

conn.close()

print(f"\n💡 Business Recommendation:")
print(f"Focus marketing efforts on growing segments and review strategy for declining products.")

QUESTION 4: PRODUCT MIX IMPACT (SQL - SINGLE QUERY)

📊 Product Mix Analysis (SQL):

📈 Product Mix Comparison (2023 vs 2024):
   policy_type  count_2023  count_2024  count_change  count_change_pct
Universal Life         5.0         1.0          -4.0             -80.0
    Whole Life         8.0         0.0          -8.0            -100.0
     Term Life        79.0        12.0         -67.0             -84.8

💰 Premium Analysis:
   policy_type  avg_premium_2023  avg_premium_2024  premium_change_pct
Universal Life            4046.0             312.0               -92.3
    Whole Life            1366.0               NaN                 NaN
     Term Life            1146.0            1613.0                40.8

📈 Market Share Analysis:
• Universal Life: 5.4% → 7.7% (+2.3pp)
• Whole Life: 8.7% → 0.0% (-8.7pp)
• Term Life: 85.9% → 92.3% (+6.4pp)

🎯 Key Insights:
• Declining products: Universal Life, Whole Life, Term Life

💡 Business Recommendation:
Focus marketing efforts on growing segments a

## Question 5: Aviation Analysis {#question-5-aviation-analysis}

**Question:** What are the differences in risk ratings, premiums, and claims performance between aviation-related occupations and non-aviation policies?

**Python Code to Answer:**

In [43]:
# Question 5: Aviation Analysis
print("=" * 60)
print("QUESTION 5: AVIATION ANALYSIS")
print("=" * 60)

# Identify aviation-related occupations
aviation_keywords = ['pilot', 'aviation', 'aircraft', 'flight', 'airline']
aviation_mask = policies_df['occupation'].str.contains('|'.join(aviation_keywords), case=False, na=False)
policies_df['is_aviation'] = aviation_mask

aviation_count = aviation_mask.sum()
total_policies = len(policies_df)

print(f"📊 Aviation Segment Overview:")
print(f"• Aviation-related policies: {aviation_count:,} ({aviation_count/total_policies*100:.1f}% of portfolio)")
print(f"• Non-aviation policies: {total_policies - aviation_count:,}")

if aviation_count > 0:
    # Compare aviation vs non-aviation segments
    segment_analysis = policies_df.groupby('is_aviation').agg({
        'policy_id': 'count',
        'premium': 'mean',
        'coverage_amount': 'mean',
        'age': 'mean'
    }).round(0)
    
    segment_analysis.index = ['Non-Aviation', 'Aviation']
    segment_analysis.columns = ['Policy_Count', 'Avg_Premium', 'Avg_Coverage', 'Avg_Age']
    
    print(f"\n📈 Segment Comparison:")
    print(segment_analysis)
    
    # Risk distribution comparison
    risk_by_segment = pd.crosstab(policies_df['is_aviation'], policies_df['risk_rating'], normalize='index') * 100
    risk_by_segment.index = ['Non-Aviation (%)', 'Aviation (%)']
    
    print(f"\n🎯 Risk Rating Distribution:")
    print(risk_by_segment.round(1))
    
    # Claims analysis if possible
    if 'claim_id' in claims_df.columns:
        policy_claims = claims_df.merge(policies_df[['policy_id', 'is_aviation']], on='policy_id', how='left')
        
        claims_by_segment = policy_claims.groupby('is_aviation').agg({
            'claim_id': 'count',
            'claim_amount': 'mean',
            'claim_status': lambda x: (x == 'Approved').mean() * 100
        }).round(1)
        
        claims_by_segment.index = ['Non-Aviation', 'Aviation']
        claims_by_segment.columns = ['Total_Claims', 'Avg_Claim_Amount', 'Approval_Rate_Pct']
        
        print(f"\n💰 Claims Performance:")
        print(claims_by_segment)

print(f"\n💡 Business Recommendation:")
print(f"Monitor aviation segment for appropriate risk-based pricing and specialized underwriting.")

QUESTION 5: AVIATION ANALYSIS
📊 Aviation Segment Overview:
• Aviation-related policies: 48 (5.4% of portfolio)
• Non-aviation policies: 842

📈 Segment Comparison:
              Policy_Count  Avg_Premium  Avg_Coverage  Avg_Age
Non-Aviation           842       2383.0      633987.0     44.0
Aviation                48       2756.0     1185833.0     42.0

🎯 Risk Rating Distribution:
risk_rating          A     B     C     D     E
Non-Aviation (%)  20.5  25.8  19.8  19.4  14.5
Aviation (%)       0.0   0.0  35.4  43.8  20.8

💰 Claims Performance:
              Total_Claims  Avg_Claim_Amount  Approval_Rate_Pct
Non-Aviation           217          498359.4               94.5
Aviation                26         1064884.6               88.5

💡 Business Recommendation:
Monitor aviation segment for appropriate risk-based pricing and specialized underwriting.

📈 Segment Comparison:
              Policy_Count  Avg_Premium  Avg_Coverage  Avg_Age
Non-Aviation           842       2383.0      633987.0     4

**SQL Code to Answer:**

In [24]:
# Question 5: Aviation Analysis - SQL Implementation (Single Query)
import duckdb

# Create fresh connection and load data
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE policies AS SELECT * FROM policies_df")
conn.execute("CREATE TABLE claims AS SELECT * FROM claims_df")

print("=" * 60)
print("QUESTION 5: AVIATION ANALYSIS (SQL - SINGLE QUERY)")
print("=" * 60)

# Compact SQL Query: Aviation vs Non-Aviation comprehensive analysis in one query
aviation_analysis_query = """
WITH segment_stats AS (
    SELECT 
        CASE WHEN LOWER(occupation) LIKE '%pilot%' OR LOWER(occupation) LIKE '%aviation%' 
             OR LOWER(occupation) LIKE '%aircraft%' OR LOWER(occupation) LIKE '%flight%' 
             OR LOWER(occupation) LIKE '%airline%' 
             THEN 'Aviation' ELSE 'Non-Aviation' END as segment,
        COUNT(*) as policy_count,
        ROUND(AVG(premium), 0) as avg_premium,
        ROUND(AVG(coverage_amount), 0) as avg_coverage,
        ROUND(AVG(age), 0) as avg_age,
        COUNT(CASE WHEN risk_rating = 'A' THEN 1 END) as rating_a,
        COUNT(CASE WHEN risk_rating = 'B' THEN 1 END) as rating_b,
        COUNT(CASE WHEN risk_rating = 'C' THEN 1 END) as rating_c,
        COUNT(CASE WHEN risk_rating = 'D' THEN 1 END) as rating_d,
        COUNT(CASE WHEN risk_rating = 'E' THEN 1 END) as rating_e
    FROM policies 
    GROUP BY 1
),
claims_stats AS (
    SELECT 
        CASE WHEN LOWER(p.occupation) LIKE '%pilot%' OR LOWER(p.occupation) LIKE '%aviation%' 
             OR LOWER(p.occupation) LIKE '%aircraft%' OR LOWER(p.occupation) LIKE '%flight%' 
             OR LOWER(p.occupation) LIKE '%airline%' 
             THEN 'Aviation' ELSE 'Non-Aviation' END as segment,
        COUNT(*) as total_claims,
        ROUND(AVG(c.claim_amount), 1) as avg_claim_amount,
        ROUND(COUNT(CASE WHEN c.claim_status = 'Approved' THEN 1 END) * 100.0 / COUNT(*), 1) as approval_rate_pct
    FROM claims c
    JOIN policies p ON c.policy_id = p.policy_id
    GROUP BY 1
),
combined_analysis AS (
    SELECT 
        s.segment,
        s.policy_count,
        s.avg_premium,
        s.avg_coverage,
        s.avg_age,
        ROUND(s.rating_a * 100.0 / s.policy_count, 1) as rating_a_pct,
        ROUND(s.rating_b * 100.0 / s.policy_count, 1) as rating_b_pct,
        ROUND(s.rating_c * 100.0 / s.policy_count, 1) as rating_c_pct,
        ROUND(s.rating_d * 100.0 / s.policy_count, 1) as rating_d_pct,
        ROUND(s.rating_e * 100.0 / s.policy_count, 1) as rating_e_pct,
        c.total_claims,
        c.avg_claim_amount,
        c.approval_rate_pct
    FROM segment_stats s
    LEFT JOIN claims_stats c ON s.segment = c.segment
)
SELECT * FROM combined_analysis ORDER BY segment;
"""

print("\n📊 Aviation Analysis (SQL):")
result = conn.execute(aviation_analysis_query).fetchdf()

print("\n📈 Segment Comparison:")
segment_cols = ['segment', 'policy_count', 'avg_premium', 'avg_coverage', 'avg_age']
segment_data = result[segment_cols].set_index('segment')
print(segment_data.to_string())

print(f"\n🎯 Risk Rating Distribution:")
print("risk_rating          A     B     C     D     E")
for _, row in result.iterrows():
    print(f"{row['segment']:12} {row['rating_a_pct']:5.1f} {row['rating_b_pct']:5.1f} {row['rating_c_pct']:5.1f} {row['rating_d_pct']:5.1f} {row['rating_e_pct']:5.1f}")

print(f"\n💰 Claims Performance:")
claims_cols = ['segment', 'total_claims', 'avg_claim_amount', 'approval_rate_pct']
claims_data = result[claims_cols].set_index('segment')
print(claims_data.to_string())

# Extract key insights
aviation_row = result[result['segment'] == 'Aviation'].iloc[0] if len(result[result['segment'] == 'Aviation']) > 0 else None
non_aviation_row = result[result['segment'] == 'Non-Aviation'].iloc[0] if len(result[result['segment'] == 'Non-Aviation']) > 0 else None

print(f"\n💡 Key Insights:")
if aviation_row is not None and non_aviation_row is not None:
    aviation_pct = round(aviation_row['policy_count'] / (aviation_row['policy_count'] + non_aviation_row['policy_count']) * 100, 1)
    print(f"• Aviation segment: {aviation_row['policy_count']} policies ({aviation_pct}% of portfolio)")
    print(f"• Higher risk profile: {aviation_row['rating_d_pct'] + aviation_row['rating_e_pct']:.1f}% vs {non_aviation_row['rating_d_pct'] + non_aviation_row['rating_e_pct']:.1f}% (D+E ratings)")
    print(f"• Premium difference: ${aviation_row['avg_premium']:,.0f} vs ${non_aviation_row['avg_premium']:,.0f} (+{(aviation_row['avg_premium']/non_aviation_row['avg_premium']-1)*100:.1f}%)")

conn.close()

print(f"\n💡 Business Recommendation:")
print(f"Monitor aviation segment for appropriate risk-based pricing and specialized underwriting.")

QUESTION 5: AVIATION ANALYSIS (SQL - SINGLE QUERY)

📊 Aviation Analysis (SQL):

📈 Segment Comparison:
              policy_count  avg_premium  avg_coverage  avg_age
segment                                                       
Aviation                48       2756.0     1185833.0     42.0
Non-Aviation           842       2383.0      633987.0     44.0

🎯 Risk Rating Distribution:
risk_rating          A     B     C     D     E
Aviation       0.0   0.0  35.4  43.8  20.8
Non-Aviation  20.5  25.8  19.8  19.4  14.5

💰 Claims Performance:
              total_claims  avg_claim_amount  approval_rate_pct
segment                                                        
Aviation                26         1064884.6               88.5
Non-Aviation           217          498359.4               94.5

💡 Key Insights:
• Aviation segment: 48 policies (5.4% of portfolio)
• Higher risk profile: 64.6% vs 33.9% (D+E ratings)
• Premium difference: $2,756 vs $2,383 (+15.7%)

💡 Business Recommendation:
Monitor 

## Question 6: Exam Exceptions {#question-6-exam-exceptions}

**Question:** How many high-risk (D/E rating) policies issued in 2024 did not require medical exams, and what are their characteristics?

**Python Code to Answer:**

In [25]:
# Question 6: Exam Exceptions
print("=" * 60)
print("QUESTION 6: EXAM EXCEPTIONS")
print("=" * 60)

# Focus on 2024 high-risk policies
high_risk_2024 = policies_df[
    (policies_df['issue_year'] == 2024) & 
    (policies_df['risk_rating'].isin(['D', 'E']))
]

print(f"📊 High-Risk Policy Analysis (2024):")
print(f"• Total high-risk (D/E) policies: {len(high_risk_2024):,}")

if len(high_risk_2024) > 0 and 'medical_exam_required' in policies_df.columns:
    # Analyze medical exam requirements
    exam_analysis = high_risk_2024.groupby(['risk_rating', 'medical_exam_required']).size().unstack(fill_value=0)
    print(f"\n🔍 Medical Exam Requirements:")
    print(exam_analysis)
    
    # Calculate exception rates
    exceptions = high_risk_2024[high_risk_2024['medical_exam_required'] == 'No']
    exception_rate = round((len(exceptions) / len(high_risk_2024) * 100), 1)
    
    print(f"\n⚠️  Exception Analysis:")
    print(f"• High-risk policies without medical exam: {len(exceptions):,}")
    print(f"• Exception rate: {exception_rate}%")
    
    if len(exceptions) > 0:
        print(f"\n📋 Exception Details:")
        exception_summary = exceptions.groupby('risk_rating').agg({
            'policy_id': 'count',
            'premium': 'mean',
            'coverage_amount': 'mean',
            'age': 'mean'
        }).round(0)
        print(exception_summary)
        
        print(f"\n🎯 Sample Exception Cases:")
        sample_cases = exceptions[['policy_id', 'risk_rating', 'age', 'premium', 'coverage_amount']].head(5)
        print(sample_cases)
    
    print(f"\n💡 Business Recommendation:")
    if exception_rate > 10:
        print(f"Exception rate of {exception_rate}% may be too high. Review underwriting guidelines.")
    else:
        print(f"Exception rate of {exception_rate}% appears reasonable. Continue monitoring.")
        
else:
    print("❌ Medical exam requirement data not available or no high-risk 2024 policies found")
    
    # Alternative analysis if medical_exam_required field is not available
    print(f"\n📋 Alternative Analysis (without medical exam field):")
    if len(high_risk_2024) > 0:
        risk_summary = high_risk_2024.groupby('risk_rating').agg({
            'policy_id': 'count',
            'premium': 'mean',
            'coverage_amount': 'mean',
            'age': 'mean'
        }).round(0)
        print(f"High-risk policy characteristics:")
        print(risk_summary)
        
        print(f"\n💡 Business Recommendation:")
        print(f"Consider implementing medical exam requirements for {len(high_risk_2024)} high-risk policies to improve risk assessment.")

QUESTION 6: EXAM EXCEPTIONS
📊 High-Risk Policy Analysis (2024):
• Total high-risk (D/E) policies: 7

🔍 Medical Exam Requirements:
medical_exam_required  No  Yes
risk_rating                   
D                       2    4
E                       0    1

⚠️  Exception Analysis:
• High-risk policies without medical exam: 2
• Exception rate: 28.6%

📋 Exception Details:
             policy_id  premium  coverage_amount   age
risk_rating                                           
D                    2    182.0          82500.0  28.0

🎯 Sample Exception Cases:
    policy_id risk_rating  age  premium  coverage_amount
198  LIF-1239           D   26    220.0         100000.0
870  LIF-1981           D   29    145.0          65000.0

💡 Business Recommendation:
Exception rate of 28.6% may be too high. Review underwriting guidelines.


**SQL Code to Answer:**

In [27]:
# Question 6: Exam Exceptions - SQL Implementation (Single Query)
import duckdb

# Create fresh connection and load data
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE policies AS SELECT * FROM policies_df")

print("=" * 60)
print("QUESTION 6: EXAM EXCEPTIONS (SQL - SINGLE QUERY)")
print("=" * 60)

# Compact SQL Query: Exam exceptions analysis for high-risk policies in one query
exam_exceptions_query = """
WITH high_risk_2024 AS (
    SELECT 
        policy_id,
        risk_rating,
        age,
        premium,
        coverage_amount,
        medical_exam_required,
        COUNT(*) OVER() as total_high_risk,
        COUNT(CASE WHEN medical_exam_required = 'No' THEN 1 END) OVER() as exceptions_count
    FROM policies
    WHERE issue_year = 2024 AND risk_rating IN ('D', 'E')
),
exam_crosstab AS (
    SELECT 
        risk_rating,
        COUNT(CASE WHEN medical_exam_required = 'No' THEN 1 END) as no_exam,
        COUNT(CASE WHEN medical_exam_required = 'Yes' THEN 1 END) as yes_exam
    FROM policies
    WHERE issue_year = 2024 AND risk_rating IN ('D', 'E')
    GROUP BY risk_rating
),
exception_details AS (
    SELECT 
        risk_rating,
        COUNT(*) as policy_count,
        ROUND(AVG(premium), 1) as avg_premium,
        ROUND(AVG(coverage_amount), 1) as avg_coverage,
        ROUND(AVG(age), 1) as avg_age
    FROM policies
    WHERE issue_year = 2024 AND risk_rating IN ('D', 'E') AND medical_exam_required = 'No'
    GROUP BY risk_rating
),
summary_stats AS (
    SELECT 
        total_high_risk,
        exceptions_count,
        ROUND(exceptions_count * 100.0 / NULLIF(total_high_risk, 0), 1) as exception_rate_pct
    FROM high_risk_2024
    LIMIT 1
)
SELECT 
    'SUMMARY' as section,
    NULL as risk_rating,
    total_high_risk as value1,
    exceptions_count as value2,
    exception_rate_pct as value3,
    NULL as value4, NULL as value5
FROM summary_stats
UNION ALL
SELECT 
    'CROSSTAB' as section,
    risk_rating,
    no_exam as value1,
    yes_exam as value2,
    NULL as value3, NULL as value4, NULL as value5
FROM exam_crosstab
UNION ALL
SELECT 
    'DETAILS' as section,
    risk_rating,
    policy_count as value1,
    avg_premium as value2,
    avg_coverage as value3,
    avg_age as value4,
    NULL as value5
FROM exception_details
ORDER BY section, risk_rating;
"""

print("\n📊 Exam Exceptions Analysis (SQL):")
result = conn.execute(exam_exceptions_query).fetchdf()

# Extract sections
summary = result[result['section'] == 'SUMMARY'].iloc[0] if len(result[result['section'] == 'SUMMARY']) > 0 else None
crosstab = result[result['section'] == 'CROSSTAB']
details = result[result['section'] == 'DETAILS']

if summary is not None:
    total_high_risk = int(summary['value1'])
    exceptions_count = int(summary['value2'])
    exception_rate = summary['value3']
    
    print(f"\n📊 High-Risk Policy Analysis (2024):")
    print(f"• Total high-risk (D/E) policies: {total_high_risk}")
    
    print(f"\n🔍 Medical Exam Requirements:")
    print("medical_exam_required  No  Yes")
    print("risk_rating                   ")
    for _, row in crosstab.iterrows():
        print(f"{row['risk_rating']:18} {int(row['value1']):3} {int(row['value2']):4}")
    
    print(f"\n⚠️  Exception Analysis:")
    print(f"• High-risk policies without medical exam: {exceptions_count}")
    print(f"• Exception rate: {exception_rate}%")
    
    if len(details) > 0:
        print(f"\n📋 Exception Details:")
        print("             policy_count  avg_premium  avg_coverage   avg_age")
        print("risk_rating                                                   ")
        for _, row in details.iterrows():
            print(f"{row['risk_rating']:11} {int(row['value1']):12} {row['value2']:11.1f} {row['value3']:12.1f} {row['value4']:9.1f}")

# Get sample exception cases
sample_exceptions_query = """
SELECT policy_id, risk_rating, age, premium, coverage_amount
FROM policies
WHERE issue_year = 2024 AND risk_rating IN ('D', 'E') AND medical_exam_required = 'No'
ORDER BY premium DESC
LIMIT 5;
"""

sample_result = conn.execute(sample_exceptions_query).fetchdf()

if len(sample_result) > 0:
    print(f"\n🎯 Sample Exception Cases:")
    print(sample_result.to_string(index=False))

print(f"\n💡 Business Recommendation:")
if summary is not None and summary['value3'] is not None:
    if summary['value3'] > 25.0:
        print(f"Exception rate of {exception_rate}% may be too high. Review underwriting guidelines.")
    else:
        print(f"Exception rate of {exception_rate}% appears reasonable for high-risk cases.")
else:
    print("Review underwriting guidelines for high-risk policies.")

conn.close()

QUESTION 6: EXAM EXCEPTIONS (SQL - SINGLE QUERY)

📊 Exam Exceptions Analysis (SQL):

📊 High-Risk Policy Analysis (2024):
• Total high-risk (D/E) policies: 7

🔍 Medical Exam Requirements:
medical_exam_required  No  Yes
risk_rating                   
D                    2    4
E                    0    1

⚠️  Exception Analysis:
• High-risk policies without medical exam: 2
• Exception rate: 28.6%

📋 Exception Details:
             policy_count  avg_premium  avg_coverage   avg_age
risk_rating                                                   
D                      2       182.5      82500.0      27.5

🎯 Sample Exception Cases:
policy_id risk_rating  age  premium  coverage_amount
 LIF-1239           D   26    220.0         100000.0
 LIF-1981           D   29    145.0          65000.0

💡 Business Recommendation:
Exception rate of 28.6% may be too high. Review underwriting guidelines.


## Question 7: Occupation Anomalies {#question-7-occupation-anomalies}

**Question:** Which occupations with at least 5 claims in 2024 have denial rates significantly different from the overall denial rate?

**Python Code to Answer:**

In [29]:
# Question 7: Occupation Anomalies
print("=" * 60)
print("QUESTION 7: OCCUPATION ANOMALIES")
print("=" * 60)

# Merge policies with claims for occupation analysis
policy_claims = claims_df.merge(policies_df[['policy_id', 'occupation']], on='policy_id', how='left')
claims_2024 = policy_claims[policy_claims['claim_year'] == 2024]

if len(claims_2024) > 0:
    # Calculate overall denial rate
    overall_denial_rate = (claims_2024['claim_status'] == 'Denied').mean()
    
    print(f"📊 Claims Analysis (2024):")
    print(f"• Total claims: {len(claims_2024):,}")
    print(f"• Overall denial rate: {overall_denial_rate:.1%}")
    
    # Analyze by occupation (minimum 5 claims)
    occupation_stats = claims_2024.groupby('occupation').agg({
        'claim_id': 'count',
        'claim_status': lambda x: (x == 'Denied').sum()
    })
    occupation_stats.columns = ['Total_Claims', 'Denied_Claims']
    occupation_stats['Denial_Rate'] = occupation_stats['Denied_Claims'] / occupation_stats['Total_Claims']
    
    # Filter occupations with ≥5 claims
    qualified_occupations = occupation_stats[occupation_stats['Total_Claims'] >= 5]
    
    print(f"\n🔍 Occupation Analysis (≥5 claims):")
    print(f"• Qualified occupations: {len(qualified_occupations)}")
    
    if len(qualified_occupations) > 0:
        # Find statistical anomalies (using simple threshold for demonstration)
        anomalies = qualified_occupations[
            (qualified_occupations['Denial_Rate'] > overall_denial_rate * 1.5) |  # 50% higher than average
            (qualified_occupations['Denial_Rate'] < overall_denial_rate * 0.5)    # 50% lower than average
        ]
        
        print(f"\n⚠️  Occupation Anomalies:")
        if len(anomalies) > 0:
            print(f"• Occupations with unusual denial rates: {len(anomalies)}")
            anomaly_summary = anomalies.sort_values('Denial_Rate', ascending=False)
            print(anomaly_summary.round(3))
        else:
            print(f"• No significant anomalies detected")
            
        print(f"\n📈 Top 10 Occupations by Claim Volume:")
        top_occupations = qualified_occupations.sort_values('Total_Claims', ascending=False).head(10)
        print(top_occupations.round(3))
        
else:
    print("❌ No 2024 claims data available for occupation analysis")

print(f"\n💡 Business Recommendation:")
print(f"Investigate occupations with unusual denial patterns and adjust underwriting criteria as needed.")

QUESTION 7: OCCUPATION ANOMALIES
📊 Claims Analysis (2024):
• Total claims: 87
• Overall denial rate: 0.0%

🔍 Occupation Analysis (≥5 claims):
• Qualified occupations: 3

⚠️  Occupation Anomalies:
• No significant anomalies detected

📈 Top 10 Occupations by Claim Volume:
                    Total_Claims  Denied_Claims  Denial_Rate
occupation                                                  
Restaurant Worker              6              0          0.0
Physical Therapist             5              0          0.0
Restaurant Manager             5              0          0.0

💡 Business Recommendation:
Investigate occupations with unusual denial patterns and adjust underwriting criteria as needed.


**SQL Code to Answer:**

In [31]:
# Question 7: Occupation Anomalies - SQL Implementation (Single Query)
import duckdb

# Create fresh connection and load data
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE policies AS SELECT * FROM policies_df")
conn.execute("CREATE TABLE claims AS SELECT * FROM claims_df")

print("=" * 60)
print("QUESTION 7: OCCUPATION ANOMALIES (SQL - SINGLE QUERY)")
print("=" * 60)

# Compact SQL Query: Occupation anomalies analysis in one query
occupation_anomalies_query = """
WITH claims_2024 AS (
    SELECT 
        c.claim_id,
        c.claim_status,
        p.occupation,
        COUNT(*) OVER() as total_claims_2024,
        COUNT(CASE WHEN c.claim_status = 'Denied' THEN 1 END) OVER() as total_denied,
        COUNT(CASE WHEN c.claim_status = 'Denied' THEN 1 END) OVER() * 1.0 / COUNT(*) OVER() as overall_denial_rate
    FROM claims c
    JOIN policies p ON c.policy_id = p.policy_id
    WHERE c.claim_year = 2024
),
occupation_stats AS (
    SELECT 
        occupation,
        COUNT(*) as total_claims,
        COUNT(CASE WHEN claim_status = 'Denied' THEN 1 END) as denied_claims,
        COUNT(CASE WHEN claim_status = 'Denied' THEN 1 END) * 1.0 / COUNT(*) as denial_rate,
        MAX(overall_denial_rate) as overall_denial_rate
    FROM claims_2024
    GROUP BY occupation
    HAVING COUNT(*) >= 5
),
anomaly_analysis AS (
    SELECT 
        occupation,
        total_claims,
        denied_claims,
        ROUND(denial_rate, 3) as denial_rate,
        ROUND(overall_denial_rate, 3) as overall_denial_rate,
        CASE 
            WHEN denial_rate > overall_denial_rate * 1.5 THEN 'HIGH_DENIAL'
            WHEN denial_rate < overall_denial_rate * 0.5 AND overall_denial_rate > 0 THEN 'LOW_DENIAL'
            ELSE 'NORMAL'
        END as anomaly_type
    FROM occupation_stats
),
summary_stats AS (
    SELECT 
        COUNT(DISTINCT occupation) as qualified_occupations,
        COUNT(CASE WHEN anomaly_type != 'NORMAL' THEN 1 END) as anomalies_count,
        MAX(c.total_claims_2024) as total_claims_2024,
        MAX(c.overall_denial_rate) as overall_denial_rate
    FROM anomaly_analysis a
    CROSS JOIN (SELECT DISTINCT total_claims_2024, overall_denial_rate FROM claims_2024) c
)
SELECT 
    'SUMMARY' as section,
    NULL as occupation,
    total_claims_2024 as value1,
    overall_denial_rate as value2,
    qualified_occupations as value3,
    anomalies_count as value4
FROM summary_stats
UNION ALL
SELECT 
    'ANOMALIES' as section,
    occupation,
    total_claims as value1,
    denied_claims as value2,
    denial_rate as value3,
    NULL as value4
FROM anomaly_analysis
WHERE anomaly_type != 'NORMAL'
UNION ALL
SELECT 
    'TOP_VOLUME' as section,
    occupation,
    total_claims as value1,
    denied_claims as value2,
    denial_rate as value3,
    NULL as value4
FROM anomaly_analysis
ORDER BY section, value1 DESC;
"""

print("\n📊 Occupation Anomalies Analysis (SQL):")
result = conn.execute(occupation_anomalies_query).fetchdf()

# Extract sections
summary = result[result['section'] == 'SUMMARY'].iloc[0] if len(result[result['section'] == 'SUMMARY']) > 0 else None
anomalies = result[result['section'] == 'ANOMALIES']
top_volume = result[result['section'] == 'TOP_VOLUME']

if summary is not None:
    total_claims = int(summary['value1']) if summary['value1'] is not None else 0
    overall_denial_rate = summary['value2'] if summary['value2'] is not None else 0.0
    qualified_occupations = int(summary['value3']) if summary['value3'] is not None else 0
    anomalies_count = int(summary['value4']) if summary['value4'] is not None else 0
    
    print(f"\n📊 Claims Analysis (2024):")
    print(f"• Total claims: {total_claims}")
    print(f"• Overall denial rate: {overall_denial_rate:.1%}")
    
    print(f"\n🔍 Occupation Analysis (≥5 claims):")
    print(f"• Qualified occupations: {qualified_occupations}")
    
    print(f"\n⚠️  Occupation Anomalies:")
    if len(anomalies) > 0:
        print(f"• Occupations with unusual denial rates: {anomalies_count}")
        print("                    Total_Claims  Denied_Claims  Denial_Rate")
        print("occupation                                                  ")
        for _, row in anomalies.iterrows():
            print(f"{row['occupation']:18} {int(row['value1']):12} {int(row['value2']):14} {row['value3']:11.3f}")
    else:
        print(f"• No significant anomalies detected")
    
    print(f"\n📈 Top 10 Occupations by Claim Volume:")
    print("                    Total_Claims  Denied_Claims  Denial_Rate")
    print("occupation                                                  ")
    
    # Show top 10 by volume
    top_10 = top_volume.head(10)
    for _, row in top_10.iterrows():
        print(f"{row['occupation']:18} {int(row['value1']):12} {int(row['value2']):14} {row['value3']:11.1f}")

print(f"\n💡 Business Recommendation:")
print(f"Investigate occupations with unusual denial patterns and adjust underwriting criteria as needed.")

conn.close()

QUESTION 7: OCCUPATION ANOMALIES (SQL - SINGLE QUERY)

📊 Occupation Anomalies Analysis (SQL):

📊 Claims Analysis (2024):
• Total claims: 86
• Overall denial rate: 0.0%

🔍 Occupation Analysis (≥5 claims):
• Qualified occupations: 3

⚠️  Occupation Anomalies:
• No significant anomalies detected

📈 Top 10 Occupations by Claim Volume:
                    Total_Claims  Denied_Claims  Denial_Rate
occupation                                                  
Restaurant Worker             6              0         0.0
Physical Therapist            5              0         0.0
Restaurant Manager            5              0         0.0

💡 Business Recommendation:
Investigate occupations with unusual denial patterns and adjust underwriting criteria as needed.


## Question 8: Claims Trends {#question-8-claims-trends}

**Question:** What are the annual and quarterly trends in claim frequency and average claim amounts, particularly for 2024?

**Python Code to Answer:**

In [32]:
# Question 8: Claims Trends
print("=" * 60)
print("QUESTION 8: CLAIMS TRENDS")
print("=" * 60)

# Analyze annual and quarterly trends
annual_trends = claims_df.groupby('claim_year').agg({
    'claim_id': 'count',
    'claim_amount': ['mean', 'sum']
}).round(0)

annual_trends.columns = ['Claim_Count', 'Avg_Amount', 'Total_Payout']

print(f"📊 Annual Claims Trends:")
print(annual_trends)

# 2024 quarterly analysis
if 2024 in claims_df['claim_year'].values:
    quarterly_2024 = claims_df[claims_df['claim_year'] == 2024].copy()
    quarterly_2024['quarter'] = quarterly_2024['claim_date'].dt.quarter
    
    quarterly_trends = quarterly_2024.groupby('quarter').agg({
        'claim_id': 'count',
        'claim_amount': 'mean'
    }).round(0)
    quarterly_trends.columns = ['Claim_Count', 'Avg_Amount']
    
    print(f"\n📈 2024 Quarterly Trends:")
    print(quarterly_trends)
    
    # Calculate trend direction
    if len(quarterly_trends) > 1:
        q1_avg = quarterly_trends.iloc[0]['Avg_Amount']
        latest_avg = quarterly_trends.iloc[-1]['Avg_Amount']
        trend_direction = "↗️ Increasing" if latest_avg > q1_avg else "↘️ Decreasing"
        change_pct = ((latest_avg - q1_avg) / q1_avg * 100).round(1)
        
        print(f"\n🎯 Trend Analysis:")
        print(f"• Direction: {trend_direction}")
        print(f"• Change from Q1: {change_pct:+.1f}%")

print(f"\n💡 Business Recommendation:")
print(f"Monitor claim trends to adjust reserves and identify emerging patterns that may impact profitability.")

QUESTION 8: CLAIMS TRENDS
📊 Annual Claims Trends:
            Claim_Count  Avg_Amount  Total_Payout
claim_year                                       
2020                  1    110000.0      110000.0
2021                 12    283625.0     3403500.0
2022                 46    519380.0    23891500.0
2023                 93    593323.0    55179000.0
2024                 82    582500.0    47765000.0
2025                  2    650000.0     1300000.0

📈 2024 Quarterly Trends:
         Claim_Count  Avg_Amount
quarter                         
1                 48    616896.0
2                 31    543032.0
3                  3    440000.0

🎯 Trend Analysis:
• Direction: ↘️ Decreasing
• Change from Q1: -28.7%

💡 Business Recommendation:
Monitor claim trends to adjust reserves and identify emerging patterns that may impact profitability.


**SQL Code to Answer:**

In [34]:
# Question 8: Claims Trends - SQL Implementation (Single Query)
import duckdb

# Create fresh connection and load data
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE claims AS SELECT * FROM claims_df")

print("=" * 60)
print("QUESTION 8: CLAIMS TRENDS (SQL - SINGLE QUERY)")
print("=" * 60)

# Compact SQL Query: Claims trends analysis in one query
claims_trends_query = """
WITH annual_trends AS (
    SELECT 
        claim_year,
        COUNT(*) as claim_count,
        ROUND(AVG(claim_amount), 1) as avg_amount,
        ROUND(SUM(claim_amount), 1) as total_payout
    FROM claims
    GROUP BY claim_year
    ORDER BY claim_year
),
quarterly_2024 AS (
    SELECT 
        EXTRACT(QUARTER FROM claim_date) as quarter,
        COUNT(*) as claim_count,
        ROUND(AVG(claim_amount), 0) as avg_amount
    FROM claims
    WHERE claim_year = 2024
    GROUP BY EXTRACT(QUARTER FROM claim_date)
    ORDER BY quarter
),
trend_analysis AS (
    SELECT 
        quarter,
        claim_count,
        avg_amount,
        LAG(claim_count) OVER (ORDER BY quarter) as prev_count,
        FIRST_VALUE(claim_count) OVER (ORDER BY quarter) as q1_count
    FROM quarterly_2024
)
SELECT 
    'ANNUAL' as section,
    claim_year as period,
    claim_count as value1,
    avg_amount as value2,
    total_payout as value3,
    NULL as value4
FROM annual_trends
UNION ALL
SELECT 
    'QUARTERLY' as section,
    quarter as period,
    claim_count as value1,
    avg_amount as value2,
    NULL as value3,
    CASE 
        WHEN q1_count > 0 THEN ROUND((claim_count - q1_count) * 100.0 / q1_count, 1)
        ELSE NULL 
    END as value4
FROM trend_analysis
ORDER BY section, period;
"""

print("\n📊 Claims Trends Analysis (SQL):")
result = conn.execute(claims_trends_query).fetchdf()

# Extract sections
annual = result[result['section'] == 'ANNUAL']
quarterly = result[result['section'] == 'QUARTERLY']

print(f"\n📊 Annual Claims Trends:")
print("            Claim_Count  Avg_Amount  Total_Payout")
print("claim_year                                       ")
for _, row in annual.iterrows():
    year = int(row['period'])
    count = int(row['value1'])
    avg_amt = row['value2']
    total = row['value3']
    print(f"{year:10} {count:12} {avg_amt:11.1f} {total:13.1f}")

print(f"\n📈 2024 Quarterly Trends:")
print("         Claim_Count  Avg_Amount")
print("quarter                         ")
for _, row in quarterly.iterrows():
    quarter = int(row['period'])
    count = int(row['value1'])
    avg_amt = row['value2']
    print(f"{quarter:7} {count:12} {avg_amt:11.1f}")

# Trend analysis
if len(quarterly) >= 2:
    q1_count = quarterly.iloc[0]['value1']
    last_count = quarterly.iloc[-1]['value1']
    change_pct = (last_count - q1_count) / q1_count * 100 if q1_count > 0 else 0
    
    print(f"\n🎯 Trend Analysis:")
    if change_pct < -10:
        direction = "↘️ Decreasing"
    elif change_pct > 10:
        direction = "↗️ Increasing"
    else:
        direction = "➡️ Stable"
    
    print(f"• Direction: {direction}")
    print(f"• Change from Q1: {change_pct:+.1f}%")

print(f"\n💡 Business Recommendation:")
print(f"Monitor claim trends to adjust reserves and identify emerging patterns that may impact profitability.")

conn.close()

QUESTION 8: CLAIMS TRENDS (SQL - SINGLE QUERY)

📊 Claims Trends Analysis (SQL):

📊 Annual Claims Trends:
            Claim_Count  Avg_Amount  Total_Payout
claim_year                                       
      2020            1    110000.0      110000.0
      2021           12    283625.0     3403500.0
      2022           46    519380.4    23891500.0
      2023           93    593322.6    55179000.0
      2024           82    582500.0    47765000.0
      2025            2    650000.0     1300000.0

📈 2024 Quarterly Trends:
         Claim_Count  Avg_Amount
quarter                         
      1           48    616896.0
      2           31    543032.0
      3            3    440000.0

🎯 Trend Analysis:
• Direction: ↘️ Decreasing
• Change from Q1: -93.8%

💡 Business Recommendation:
Monitor claim trends to adjust reserves and identify emerging patterns that may impact profitability.


## Question 9: Pre-existing Conditions {#question-9-preexisting-conditions}

**Question:** How do pre-existing conditions impact claim denial rates and average claim amounts compared to claims without pre-existing conditions?

**Python Code to Answer:**

In [35]:
# Question 9: Pre-existing Conditions Impact
print("=" * 60)
print("QUESTION 9: PRE-EXISTING CONDITIONS")
print("=" * 60)

if 'pre_existing_condition' in claims_df.columns:
    condition_analysis = claims_df.groupby('pre_existing_condition').agg({
        'claim_id': 'count',
        'claim_amount': 'mean',
        'claim_status': lambda x: (x == 'Denied').mean() * 100
    }).round(1)
    
    condition_analysis.columns = ['Total_Claims', 'Avg_Amount', 'Denial_Rate_Pct']
    condition_analysis.index = ['No Pre-existing', 'Has Pre-existing']
    
    print(f"📊 Pre-existing Condition Impact:")
    print(condition_analysis)
    
    # Calculate relative risk
    if len(condition_analysis) == 2:
        risk_ratio = (condition_analysis.loc['Has Pre-existing', 'Denial_Rate_Pct'] / 
                     condition_analysis.loc['No Pre-existing', 'Denial_Rate_Pct'])
        
        print(f"\n🎯 Risk Analysis:")
        print(f"• Denial rate - No condition: {condition_analysis.loc['No Pre-existing', 'Denial_Rate_Pct']:.1f}%")
        print(f"• Denial rate - Has condition: {condition_analysis.loc['Has Pre-existing', 'Denial_Rate_Pct']:.1f}%")
        print(f"• Relative risk: {risk_ratio:.2f}x higher with pre-existing conditions")
        
else:
    print("❌ Pre-existing condition data not available")

print(f"\n💡 Business Recommendation:")
print(f"Ensure proper disclosure and risk assessment for pre-existing conditions during underwriting.")

QUESTION 9: PRE-EXISTING CONDITIONS
📊 Pre-existing Condition Impact:
                  Total_Claims  Avg_Amount  Denial_Rate_Pct
No Pre-existing            169    668331.4              5.9
Has Pre-existing            67    279119.4              3.0

🎯 Risk Analysis:
• Denial rate - No condition: 5.9%
• Denial rate - Has condition: 3.0%
• Relative risk: 0.51x higher with pre-existing conditions

💡 Business Recommendation:
Ensure proper disclosure and risk assessment for pre-existing conditions during underwriting.


**SQL Code to Answer:**

In [38]:
# Question 9: Pre-existing Conditions - SQL Implementation (Single Query)
import duckdb

# Create fresh connection and load data
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE claims AS SELECT * FROM claims_df")

print("=" * 60)
print("QUESTION 9: PRE-EXISTING CONDITIONS (SQL - SINGLE QUERY)")
print("=" * 60)

# Check if pre_existing_condition column exists
has_preexisting_col = conn.execute("""
    SELECT column_name 
    FROM information_schema.columns 
    WHERE table_name = 'claims' AND column_name = 'pre_existing_condition'
""").fetchall()

if len(has_preexisting_col) > 0:
    # Compact SQL Query: Pre-existing conditions analysis in one query
    preexisting_conditions_query = """
    WITH condition_analysis AS (
        SELECT 
            CASE 
                WHEN pre_existing_condition = 'Yes' THEN 'Has Pre-existing'
                ELSE 'No Pre-existing'
            END as condition_status,
            COUNT(*) as total_claims,
            ROUND(AVG(claim_amount), 1) as avg_amount,
            ROUND(COUNT(CASE WHEN claim_status = 'Denied' THEN 1 END) * 100.0 / COUNT(*), 1) as denial_rate_pct
        FROM claims
        WHERE pre_existing_condition IS NOT NULL
        GROUP BY 
            CASE 
                WHEN pre_existing_condition = 'Yes' THEN 'Has Pre-existing'
                ELSE 'No Pre-existing'
            END
    ),
    risk_calculation AS (
        SELECT 
            *,
            LAG(denial_rate_pct) OVER (ORDER BY condition_status) as prev_denial_rate
        FROM condition_analysis
        ORDER BY condition_status
    )
    SELECT 
        condition_status,
        total_claims,
        avg_amount,
        denial_rate_pct,
        CASE 
            WHEN prev_denial_rate > 0 AND prev_denial_rate IS NOT NULL 
            THEN ROUND(denial_rate_pct / prev_denial_rate, 2)
            ELSE NULL 
        END as relative_risk
    FROM risk_calculation
    ORDER BY condition_status DESC;
    """
    
    print("\n📊 Pre-existing Conditions Analysis (SQL):")
    result = conn.execute(preexisting_conditions_query).fetchdf()
    
    if len(result) > 0:
        print(f"\n📊 Pre-existing Condition Impact:")
        print("                  Total_Claims  Avg_Amount  Denial_Rate_Pct")
        for _, row in result.iterrows():
            print(f"{row['condition_status']:17} {int(row['total_claims']):12} {row['avg_amount']:11.1f} {row['denial_rate_pct']:15.1f}")
        
        # Calculate and display risk analysis
        if len(result) == 2:
            has_condition = result[result['condition_status'] == 'Has Pre-existing'].iloc[0]
            no_condition = result[result['condition_status'] == 'No Pre-existing'].iloc[0]
            
            risk_ratio = has_condition['denial_rate_pct'] / no_condition['denial_rate_pct'] if no_condition['denial_rate_pct'] > 0 else 0
            
            print(f"\n🎯 Risk Analysis:")
            print(f"• Denial rate - No condition: {no_condition['denial_rate_pct']:.1f}%")
            print(f"• Denial rate - Has condition: {has_condition['denial_rate_pct']:.1f}%")
            print(f"• Relative risk: {risk_ratio:.2f}x higher with pre-existing conditions")
    else:
        print("❌ No pre-existing condition data available for analysis")

else:
    print("❌ Pre-existing condition data not available")

print(f"\n💡 Business Recommendation:")
print(f"Ensure proper disclosure and risk assessment for pre-existing conditions during underwriting.")

conn.close()

QUESTION 9: PRE-EXISTING CONDITIONS (SQL - SINGLE QUERY)

📊 Pre-existing Conditions Analysis (SQL):

📊 Pre-existing Condition Impact:
                  Total_Claims  Avg_Amount  Denial_Rate_Pct
No Pre-existing            169    668331.4             5.9
Has Pre-existing            67    279119.4             3.0

🎯 Risk Analysis:
• Denial rate - No condition: 5.9%
• Denial rate - Has condition: 3.0%
• Relative risk: 0.51x higher with pre-existing conditions

💡 Business Recommendation:
Ensure proper disclosure and risk assessment for pre-existing conditions during underwriting.


## Question 10: Data Quality {#question-10-data-quality}

**Question:** What percentage of critical fields are missing in both policy and claims data, and how does this affect data completeness scores?

**Python Code to Answer:**

In [40]:
# Question 10: Data Quality Assessment
print("=" * 60)
print("QUESTION 10: DATA QUALITY")
print("=" * 60)

print(f"📊 Data Quality Assessment:")

# Check for missing values in critical fields
critical_policy_fields = ['premium', 'coverage_amount', 'risk_rating', 'age']
critical_claims_fields = ['claim_amount', 'claim_status', 'claim_date']

print(f"\n🔍 Missing Data Analysis:")
print(f"Policy Data:")
for field in critical_policy_fields:
    if field in policies_df.columns:
        missing_count = policies_df[field].isna().sum()
        missing_pct = (missing_count / len(policies_df) * 100).round(1)
        print(f"• {field}: {missing_count:,} missing ({missing_pct}%)")

print(f"\nClaims Data:")
for field in critical_claims_fields:
    if field in claims_df.columns:
        missing_count = claims_df[field].isna().sum()
        missing_pct = (missing_count / len(claims_df) * 100).round(1)
        print(f"• {field}: {missing_count:,} missing ({missing_pct}%)")

# Data completeness score
total_policy_cells = len(policies_df) * len(critical_policy_fields)
total_claims_cells = len(claims_df) * len(critical_claims_fields)

policy_missing = sum(policies_df[field].isna().sum() for field in critical_policy_fields if field in policies_df.columns)
claims_missing = sum(claims_df[field].isna().sum() for field in critical_claims_fields if field in claims_df.columns)

policy_completeness = ((total_policy_cells - policy_missing) / total_policy_cells * 100).round(1)
claims_completeness = ((total_claims_cells - claims_missing) / total_claims_cells * 100).round(1)

print(f"\n📈 Overall Data Quality:")
print(f"• Policy data completeness: {policy_completeness}%")
print(f"• Claims data completeness: {claims_completeness}%")

quality_threshold = 95.0
if policy_completeness >= quality_threshold and claims_completeness >= quality_threshold:
    print(f"✅ Data quality meets standards (≥{quality_threshold}%)")
else:
    print(f"⚠️  Data quality below standards (<{quality_threshold}%)")

print(f"\n💡 Business Recommendation:")
print(f"Focus on improving data capture for fields with >5% missing values.")

print("\n" + "="*80)
print("📋 ANALYSIS COMPLETE - ALL 10 QUESTIONS ANSWERED")
print("="*80)

QUESTION 10: DATA QUALITY
📊 Data Quality Assessment:

🔍 Missing Data Analysis:
Policy Data:
• premium: 0 missing (0.0%)
• coverage_amount: 0 missing (0.0%)
• risk_rating: 0 missing (0.0%)
• age: 0 missing (0.0%)

Claims Data:
• claim_amount: 0 missing (0.0%)
• claim_status: 0 missing (0.0%)
• claim_date: 0 missing (0.0%)

📈 Overall Data Quality:
• Policy data completeness: 100.0%
• Claims data completeness: 100.0%
✅ Data quality meets standards (≥95.0%)

💡 Business Recommendation:
Focus on improving data capture for fields with >5% missing values.

📋 ANALYSIS COMPLETE - ALL 10 QUESTIONS ANSWERED


**SQL Code to Answer:**

In [41]:
# Question 10: Data Quality - SQL Implementation (Single Query)
import duckdb

# Create fresh connection and load data
conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE policies AS SELECT * FROM policies_df")
conn.execute("CREATE TABLE claims AS SELECT * FROM claims_df")

print("=" * 60)
print("QUESTION 10: DATA QUALITY (SQL - SINGLE QUERY)")
print("=" * 60)

# Compact SQL Query: Data quality analysis in one query
data_quality_query = """
WITH policy_quality AS (
    SELECT 
        'Policy Data' as data_source,
        'premium' as field_name,
        COUNT(*) as total_records,
        COUNT(premium) as non_null_records,
        COUNT(*) - COUNT(premium) as missing_count,
        ROUND((COUNT(*) - COUNT(premium)) * 100.0 / COUNT(*), 1) as missing_pct
    FROM policies
    UNION ALL
    SELECT 
        'Policy Data' as data_source,
        'coverage_amount' as field_name,
        COUNT(*) as total_records,
        COUNT(coverage_amount) as non_null_records,
        COUNT(*) - COUNT(coverage_amount) as missing_count,
        ROUND((COUNT(*) - COUNT(coverage_amount)) * 100.0 / COUNT(*), 1) as missing_pct
    FROM policies
    UNION ALL
    SELECT 
        'Policy Data' as data_source,
        'risk_rating' as field_name,
        COUNT(*) as total_records,
        COUNT(risk_rating) as non_null_records,
        COUNT(*) - COUNT(risk_rating) as missing_count,
        ROUND((COUNT(*) - COUNT(risk_rating)) * 100.0 / COUNT(*), 1) as missing_pct
    FROM policies
    UNION ALL
    SELECT 
        'Policy Data' as data_source,
        'age' as field_name,
        COUNT(*) as total_records,
        COUNT(age) as non_null_records,
        COUNT(*) - COUNT(age) as missing_count,
        ROUND((COUNT(*) - COUNT(age)) * 100.0 / COUNT(*), 1) as missing_pct
    FROM policies
),
claims_quality AS (
    SELECT 
        'Claims Data' as data_source,
        'claim_amount' as field_name,
        COUNT(*) as total_records,
        COUNT(claim_amount) as non_null_records,
        COUNT(*) - COUNT(claim_amount) as missing_count,
        ROUND((COUNT(*) - COUNT(claim_amount)) * 100.0 / COUNT(*), 1) as missing_pct
    FROM claims
    UNION ALL
    SELECT 
        'Claims Data' as data_source,
        'claim_status' as field_name,
        COUNT(*) as total_records,
        COUNT(claim_status) as non_null_records,
        COUNT(*) - COUNT(claim_status) as missing_count,
        ROUND((COUNT(*) - COUNT(claim_status)) * 100.0 / COUNT(*), 1) as missing_pct
    FROM claims
    UNION ALL
    SELECT 
        'Claims Data' as data_source,
        'claim_date' as field_name,
        COUNT(*) as total_records,
        COUNT(claim_date) as non_null_records,
        COUNT(*) - COUNT(claim_date) as missing_count,
        ROUND((COUNT(*) - COUNT(claim_date)) * 100.0 / COUNT(*), 1) as missing_pct
    FROM claims
),
completeness_summary AS (
    SELECT 
        data_source,
        COUNT(*) as field_count,
        SUM(total_records) as total_cells,
        SUM(missing_count) as missing_cells,
        ROUND((SUM(total_records) - SUM(missing_count)) * 100.0 / SUM(total_records), 1) as completeness_pct
    FROM (
        SELECT * FROM policy_quality
        UNION ALL
        SELECT * FROM claims_quality
    ) combined
    GROUP BY data_source
)
SELECT 
    'FIELDS' as section,
    data_source,
    field_name,
    missing_count as value1,
    missing_pct as value2,
    NULL as value3
FROM (
    SELECT * FROM policy_quality
    UNION ALL
    SELECT * FROM claims_quality
) all_fields
UNION ALL
SELECT 
    'SUMMARY' as section,
    data_source,
    NULL as field_name,
    completeness_pct as value1,
    NULL as value2,
    CASE WHEN completeness_pct >= 95.0 THEN 1 ELSE 0 END as value3
FROM completeness_summary
ORDER BY section, data_source, field_name;
"""

print("\n📊 Data Quality Analysis (SQL):")
result = conn.execute(data_quality_query).fetchdf()

# Extract sections
fields = result[result['section'] == 'FIELDS']
summary = result[result['section'] == 'SUMMARY']

print(f"\n🔍 Missing Data Analysis:")

# Policy Data
policy_fields = fields[fields['data_source'] == 'Policy Data']
print(f"Policy Data:")
for _, row in policy_fields.iterrows():
    field = row['field_name']
    missing = int(row['value1'])
    pct = row['value2']
    print(f"• {field}: {missing} missing ({pct}%)")

# Claims Data
claims_fields = fields[fields['data_source'] == 'Claims Data']
print(f"\nClaims Data:")
for _, row in claims_fields.iterrows():
    field = row['field_name']
    missing = int(row['value1'])
    pct = row['value2']
    print(f"• {field}: {missing} missing ({pct}%)")

print(f"\n📈 Overall Data Quality:")
quality_threshold = 95.0
all_meet_standards = True

for _, row in summary.iterrows():
    data_source = row['data_source']
    completeness = row['value1']
    meets_standards = row['value3'] == 1
    
    print(f"• {data_source} completeness: {completeness}%")
    if not meets_standards:
        all_meet_standards = False

if all_meet_standards:
    print(f"✅ Data quality meets standards (≥{quality_threshold}%)")
else:
    print(f"⚠️  Data quality below standards (<{quality_threshold}%)")

print(f"\n💡 Business Recommendation:")
print(f"Focus on improving data capture for fields with >5% missing values.")

print("\n" + "="*80)
print("📋 ANALYSIS COMPLETE - ALL 10 QUESTIONS ANSWERED WITH SQL")
print("="*80)

conn.close()

QUESTION 10: DATA QUALITY (SQL - SINGLE QUERY)

📊 Data Quality Analysis (SQL):

🔍 Missing Data Analysis:
Policy Data:
• age: 0 missing (0.0%)
• coverage_amount: 0 missing (0.0%)
• premium: 0 missing (0.0%)
• risk_rating: 0 missing (0.0%)

Claims Data:
• claim_amount: 0 missing (0.0%)
• claim_date: 0 missing (0.0%)
• claim_status: 0 missing (0.0%)

📈 Overall Data Quality:
• Claims Data completeness: 100.0%
• Policy Data completeness: 100.0%
✅ Data quality meets standards (≥95.0%)

💡 Business Recommendation:
Focus on improving data capture for fields with >5% missing values.

📋 ANALYSIS COMPLETE - ALL 10 QUESTIONS ANSWERED WITH SQL


QUESTION 10: DATA QUALITY
📊 Data Quality Assessment:

🔍 Missing Data Analysis:
Policy Data:
• premium: 0 missing (0.0%)
• coverage_amount: 0 missing (0.0%)
• risk_rating: 0 missing (0.0%)
• age: 0 missing (0.0%)

Claims Data:
• claim_amount: 0 missing (0.0%)
• claim_status: 0 missing (0.0%)
• claim_date: 0 missing (0.0%)

📈 Overall Data Quality:
• Policy data completeness: 100.0%
• Claims data completeness: 100.0%
✅ Data quality meets standards (≥95.0%)

💡 Business Recommendation:
Focus on improving data capture for fields with >5% missing values.

📋 ANALYSIS COMPLETE - ALL 10 QUESTIONS ANSWERED
